# Cortex AI Guardrails Usage Analysis
*Co-authored with CoCo*

This notebook analyzes the `CORTEX_AI_GUARDRAILS_USAGE_HISTORY` view covering:
- Recent scan activity & prompt injection detection
- Credit/token consumption by source
- Tool-type breakdown & trend analysis

## 1. Recent Guardrail Scans (72h)

In [ ]:
%%sql -r recent_scans
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('hour', -72, CURRENT_TIMESTAMP())
ORDER BY USAGE_TIME DESC
LIMIT 100;

## 2. Flagged Prompt Injection Attempts

In [ ]:
%%sql -r flagged_events
SELECT USER_NAME, AGENTIC_SOURCE, REQUEST_ID, USAGE_TIME, TOKENS, TOKEN_CREDITS, GUARDRAIL_RESULTS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE GUARDRAILS_SIGNAL = TRUE
  AND USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY USAGE_TIME DESC;

## 3. Usage by Agentic Source

In [ ]:
%%sql -r source_summary
SELECT
    AGENTIC_SOURCE,
    COUNT(*) AS scan_count,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(TOKENS) AS total_tokens,
    SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flagged_count,
    ROUND(flagged_count / NULLIF(scan_count, 0) * 100, 2) AS flag_rate_pct
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY AGENTIC_SOURCE
ORDER BY total_credits DESC;

## 4. Hourly Scan Trend (3 days)

In [ ]:
%%sql -r hourly_trend
SELECT
    DATE_TRUNC('hour', USAGE_TIME) AS scan_hour,
    COUNT(*) AS scan_count,
    SUM(TOKEN_CREDITS) AS hourly_credits,
    SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flagged_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -3, CURRENT_TIMESTAMP())
GROUP BY scan_hour
ORDER BY scan_hour;

## 5. Granular Token Breakdown

In [ ]:
%%sql -r token_breakdown
SELECT
    AGENTIC_SOURCE,
    SUM(TOKENS_GRANULAR:input::NUMBER) AS input_tokens,
    SUM(TOKENS_GRANULAR:output::NUMBER) AS output_tokens,
    SUM(TOKENS_GRANULAR:cache_read_input::NUMBER) AS cache_read_tokens,
    SUM(TOKENS_GRANULAR:cache_write_input::NUMBER) AS cache_write_tokens
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY AGENTIC_SOURCE
ORDER BY input_tokens DESC;

## 6. Tool Type Analysis (Flatten GUARDRAIL_RESULTS)

In [ ]:
%%sql -r tool_analysis
SELECT
    gr.value:tool_type::VARCHAR AS tool_type,
    COUNT(*) AS scan_count,
    SUM(CASE WHEN gr.value:indirect_prompt_injection::BOOLEAN = TRUE THEN 1 ELSE 0 END) AS injection_detected,
    AVG(gr.value:token_count::NUMBER) AS avg_token_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY h,
    LATERAL FLATTEN(input => h.GUARDRAIL_RESULTS) gr
WHERE h.USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY tool_type
ORDER BY scan_count DESC;

## 7. Top Users & Role Audit

In [ ]:
%%sql -r top_users
SELECT USER_NAME, COUNT(*) AS scan_count, SUM(TOKEN_CREDITS) AS total_credits,
    SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flagged_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY USER_NAME ORDER BY scan_count DESC LIMIT 20;

In [ ]:
%%sql -r role_audit
SELECT METADATA:role_name::VARCHAR AS role_name, COUNT(*) AS scan_count,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flagged_count
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY role_name ORDER BY scan_count DESC;

## 8. Daily Summary (30 days)

In [ ]:
%%sql -r daily_summary
SELECT
    DATE_TRUNC('day', USAGE_TIME) AS scan_date,
    COUNT(*) AS total_scans,
    COUNT(DISTINCT USER_NAME) AS unique_users,
    SUM(TOKEN_CREDITS) AS daily_credits,
    SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS daily_flagged,
    ROUND(daily_flagged / NULLIF(total_scans, 0) * 100, 2) AS flag_rate_pct
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
GROUP BY scan_date
ORDER BY scan_date DESC;

---
## 9. Privileged Role Injection Analysis
Identify injection flags from ACCOUNTADMIN, SYSADMIN, SECURITYADMIN — highest severity events.

In [ ]:
%%sql -r privileged_flags
SELECT
    USER_NAME,
    METADATA:role_name::VARCHAR AS role_name,
    AGENTIC_SOURCE,
    USAGE_TIME,
    REQUEST_ID,
    TOKEN_CREDITS,
    GUARDRAIL_RESULTS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE GUARDRAILS_SIGNAL = TRUE
  AND METADATA:role_name::VARCHAR IN ('ACCOUNTADMIN', 'SYSADMIN', 'SECURITYADMIN')
  AND USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY USAGE_TIME DESC;

## 10. Off-Hours Flagged Activity
Scans flagged outside business hours (before 08:00 or after 20:00) — potential credential compromise indicator.

In [ ]:
%%sql -r off_hours_flags
SELECT
    USER_NAME,
    METADATA:role_name::VARCHAR AS role_name,
    AGENTIC_SOURCE,
    USAGE_TIME,
    HOUR(USAGE_TIME) AS scan_hour,
    TOKEN_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE GUARDRAILS_SIGNAL = TRUE
  AND (HOUR(USAGE_TIME) < 8 OR HOUR(USAGE_TIME) >= 20)
  AND USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
ORDER BY USAGE_TIME DESC;

## 11. Velocity Spike Detection
Users with hourly scan counts exceeding 3x their average — possible automated attack or compromised bot.

In [ ]:
%%sql -r velocity_spikes
WITH hourly_counts AS (
    SELECT
        USER_NAME,
        DATE_TRUNC('hour', USAGE_TIME) AS scan_hour,
        COUNT(*) AS hourly_count
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY USER_NAME, scan_hour
),
baselines AS (
    SELECT USER_NAME, AVG(hourly_count) AS avg_hourly, STDDEV(hourly_count) AS std_hourly
    FROM hourly_counts
    GROUP BY USER_NAME
    HAVING COUNT(*) >= 5
)
SELECT
    h.USER_NAME,
    h.scan_hour,
    h.hourly_count,
    ROUND(b.avg_hourly, 1) AS baseline_avg,
    ROUND(h.hourly_count / NULLIF(b.avg_hourly, 0), 1) AS spike_multiplier
FROM hourly_counts h
JOIN baselines b ON h.USER_NAME = b.USER_NAME
WHERE h.hourly_count >= b.avg_hourly * 3
  AND h.hourly_count >= 10
ORDER BY spike_multiplier DESC
LIMIT 20;

## 12. Nested Agent Chain Analysis
Requests with PARENT_REQUEST_ID — multi-hop agent calls where injection could propagate across boundaries.

In [ ]:
%%sql -r nested_chains
SELECT
    PARENT_REQUEST_ID,
    COUNT(*) AS child_scans,
    COUNT(DISTINCT USER_NAME) AS unique_users,
    ARRAY_AGG(DISTINCT AGENTIC_SOURCE) AS sources,
    SUM(TOKEN_CREDITS) AS total_credits,
    SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flagged_in_chain,
    MAX(USAGE_TIME) AS last_activity
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE PARENT_REQUEST_ID IS NOT NULL
  AND USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY PARENT_REQUEST_ID
HAVING child_scans > 1
ORDER BY flagged_in_chain DESC, child_scans DESC
LIMIT 30;

## 13. Credit Cost Forecast
Daily credit consumption trend with 30-day projection.

In [ ]:
%%sql -r credit_forecast
WITH daily_costs AS (
    SELECT
        DATE_TRUNC('day', USAGE_TIME) AS cost_date,
        SUM(TOKEN_CREDITS) AS daily_credits,
        SUM(TOKENS) AS daily_tokens,
        COUNT(*) AS daily_scans
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
    GROUP BY cost_date
)
SELECT
    cost_date,
    daily_credits,
    daily_tokens,
    daily_scans,
    AVG(daily_credits) OVER (ORDER BY cost_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rolling_7d_avg_credits,
    SUM(daily_credits) OVER (ORDER BY cost_date) AS cumulative_credits
FROM daily_costs
ORDER BY cost_date;

## 14. Token Cache Efficiency
Measure cache hit ratio across sources — higher cache reads = lower cost per scan.

In [ ]:
%%sql -r cache_efficiency
SELECT
    AGENTIC_SOURCE,
    COUNT(*) AS scan_count,
    SUM(TOKENS_GRANULAR:input::NUMBER) AS total_input,
    SUM(TOKENS_GRANULAR:cache_read_input::NUMBER) AS total_cache_read,
    SUM(TOKENS_GRANULAR:cache_write_input::NUMBER) AS total_cache_write,
    SUM(TOKENS_GRANULAR:output::NUMBER) AS total_output,
    ROUND(
        SUM(TOKENS_GRANULAR:cache_read_input::NUMBER) /
        NULLIF(SUM(TOKENS_GRANULAR:input::NUMBER) + SUM(TOKENS_GRANULAR:cache_read_input::NUMBER) + SUM(TOKENS_GRANULAR:output::NUMBER), 0)
        * 100, 2
    ) AS cache_hit_pct
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY AGENTIC_SOURCE
ORDER BY cache_hit_pct DESC;

## 15. User Risk Score (SQL-based)
Composite risk ranking: weighted flag rate + volume + off-hours + role diversity.

In [ ]:
%%sql -r user_risk_scores
WITH user_stats AS (
    SELECT
        USER_NAME,
        COUNT(*) AS scan_count,
        SUM(TOKEN_CREDITS) AS total_credits,
        SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flagged_count,
        SUM(CASE WHEN HOUR(USAGE_TIME) < 8 OR HOUR(USAGE_TIME) >= 20 THEN 1 ELSE 0 END) AS off_hours_count,
        COUNT(DISTINCT METADATA:role_name::VARCHAR) AS unique_roles
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
    GROUP BY USER_NAME
    HAVING scan_count >= 5
)
SELECT
    USER_NAME,
    scan_count,
    flagged_count,
    ROUND(flagged_count / scan_count * 100, 2) AS flag_rate_pct,
    off_hours_count,
    unique_roles,
    total_credits,
    -- Composite risk score (0-100)
    ROUND(
        LEAST(flagged_count / NULLIF(scan_count, 0) * 100, 100) * 0.30
        + LEAST(flagged_count / NULLIF((SELECT MAX(flagged_count) FROM user_stats), 1) * 100, 100) * 0.25
        + LEAST(off_hours_count / NULLIF(scan_count, 0) * 100 * 2, 100) * 0.20
        + LEAST((unique_roles - 1) * 25, 100) * 0.15
        + LEAST(total_credits / NULLIF((SELECT MAX(total_credits) FROM user_stats), 0.001) * 50, 100) * 0.10
    , 1) AS risk_score
FROM user_stats
ORDER BY risk_score DESC
LIMIT 15;

## 16. Drift Detection (Week-over-Week)
Compare this week vs last week to detect security posture degradation.

In [ ]:
%%sql -r week_over_week_drift
WITH this_week AS (
    SELECT
        COUNT(*) AS scans,
        SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flags,
        SUM(TOKEN_CREDITS) AS credits
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
),
last_week AS (
    SELECT
        COUNT(*) AS scans,
        SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS flags,
        SUM(TOKEN_CREDITS) AS credits
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
    WHERE USAGE_TIME BETWEEN DATEADD('day', -14, CURRENT_TIMESTAMP())
                         AND DATEADD('day', -7, CURRENT_TIMESTAMP())
)
SELECT
    'This Week' AS period,
    t.scans, t.flags,
    ROUND(t.flags / NULLIF(t.scans, 0) * 100, 4) AS flag_rate_pct,
    t.credits,
    NULL AS scan_change_pct,
    NULL AS flag_rate_change_pct
FROM this_week t
UNION ALL
SELECT
    'Last Week', l.scans, l.flags,
    ROUND(l.flags / NULLIF(l.scans, 0) * 100, 4),
    l.credits, NULL, NULL
FROM last_week l
UNION ALL
SELECT
    'Delta (%)',
    NULL, NULL, NULL, NULL,
    ROUND((t.scans - l.scans) / NULLIF(l.scans, 0) * 100, 1),
    ROUND(
        (t.flags / NULLIF(t.scans, 0) * 100 - l.flags / NULLIF(l.scans, 0) * 100)
        / NULLIF(l.flags / NULLIF(l.scans, 0) * 100, 0.01) * 100
    , 1)
FROM this_week t, last_week l;

## 17. MITRE ATLAS Detection Summary
Map guardrail signals to AI threat framework techniques.

In [ ]:
%%sql -r mitre_detections
WITH tool_details AS (
    SELECT
        h.REQUEST_ID,
        h.USER_NAME,
        h.USAGE_TIME,
        h.GUARDRAILS_SIGNAL,
        gr.value:tool_type::VARCHAR AS tool_type,
        gr.value:indirect_prompt_injection::BOOLEAN AS is_indirect
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY h,
        LATERAL FLATTEN(input => h.GUARDRAIL_RESULTS) gr
    WHERE h.USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
)
SELECT
    'AML.T0051' AS ttp_id,
    'LLM Prompt Injection' AS technique,
    'Critical' AS severity,
    COUNT(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 END) AS detection_count,
    MAX(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN USAGE_TIME END) AS last_detected
FROM tool_details
UNION ALL
SELECT
    'AML.T0051.001', 'Indirect Prompt Injection', 'Critical',
    COUNT(CASE WHEN is_indirect = TRUE THEN 1 END),
    MAX(CASE WHEN is_indirect = TRUE THEN USAGE_TIME END)
FROM tool_details
UNION ALL
SELECT
    'AML.T0051.002', 'Tool Output Manipulation', 'High',
    COUNT(CASE WHEN is_indirect = TRUE AND tool_type IN ('web_search', 'server_mcp') THEN 1 END),
    MAX(CASE WHEN is_indirect = TRUE AND tool_type IN ('web_search', 'server_mcp') THEN USAGE_TIME END)
FROM tool_details;

## 18. Compliance Control Checks
Automated pass/fail checks against BFSI regulatory controls.

In [ ]:
%%sql -r compliance_checks
WITH base AS (
    SELECT
        COUNT(*) AS total_scans,
        SUM(CASE WHEN GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS total_flags,
        SUM(TOKEN_CREDITS) AS total_credits,
        SUM(CASE WHEN METADATA:role_name::VARCHAR IN ('ACCOUNTADMIN','SYSADMIN','SECURITYADMIN')
                 AND GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS priv_flags,
        SUM(CASE WHEN (HOUR(USAGE_TIME) < 8 OR HOUR(USAGE_TIME) >= 20)
                 AND GUARDRAILS_SIGNAL = TRUE THEN 1 ELSE 0 END) AS off_hours_flags
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_GUARDRAILS_USAGE_HISTORY
    WHERE USAGE_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
)
SELECT 'PAM-001' AS control, 'No privileged injection flags' AS description,
    IFF(priv_flags = 0, 'PASS', 'FAIL') AS status, priv_flags || ' violations' AS evidence, 'SOX / ISAE 3402' AS framework
FROM base
UNION ALL
SELECT 'SEC-001', 'Flag rate < 15%',
    IFF(total_flags / NULLIF(total_scans, 0) * 100 < 15, 'PASS', 'FAIL'),
    ROUND(total_flags / NULLIF(total_scans, 0) * 100, 2) || '%', 'NIST AI RMF'
FROM base
UNION ALL
SELECT 'FIN-001', 'Daily credits < 1.0',
    IFF(total_credits / 7 < 1.0, 'PASS', 'FAIL'),
    ROUND(total_credits / 7, 6) || ' credits/day', 'Internal Risk'
FROM base
UNION ALL
SELECT 'ACC-001', 'No off-hours flags',
    IFF(off_hours_flags = 0, 'PASS', 'FAIL'),
    off_hours_flags || ' violations', 'ISO 27001 / PCI-DSS'
FROM base
UNION ALL
SELECT 'GOV-001', 'Guardrails enabled',
    IFF(total_scans > 0, 'PASS', 'NO DATA'),
    total_scans || ' scans recorded', 'EU AI Act / DORA'
FROM base;